#   AML - TOP GENE SENSITIVITY ANALYSIS WITH DEPRIVATION INDEX
This script performs logistic regression and Cox regression using the rare variant polygenic risk scores for the top genes. Covariates for the logistic regression include age, sex, ten principal components, exome burden, and either the electronic health record index date or the observation period. Covariates for the Cox regression—modeling time from birth or time from the first electronic health record—include sex, ten principal components, exome burden, year of birth or year of first electronic health record, and optionally the electronic health record index date. For this analysis, additional covariates were included in the model to evaluate whether their inclusion would improve the significance of the top genes. In particular, both the overall deprivation index and its individual components were incorporated as covariates.

In [ ]:
# Load packages 
library(data.table)
library(tidyverse)
library(readr)
library(survival)
library(ggplot2)
library(forcats)
library(janitor)

In [ ]:
# Ensure a clean directory for the logistic regression loop 
system("rm -rf burden_polygenic_score")
system("mkdir -p burden_polygenic_score")
system("rm -rf logistic_summary")
system("mkdir -p logistic_summary")

In [ ]:
# Copy gene-set burden score files to directory 
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gene_burden_analysis/burden_gene_sets/top_genes/*prs.tsv burden_polygenic_score")

In [ ]:
# List of file present in the burden polygenic score folder 
files <- list.files(path="burden_polygenic_score")

In [ ]:
# Load files necessary for logistic regeression 
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_phenotype_files/aml_phenotype_files/EUR_phenotype_file.txt burden_polygenic_score")
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/phenotype_files/aml_phenotype_time_to_event_with_covariates.tsv burden_polygenic_score")
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/gene_burden_analysis/burden_gene_sets/exome/* burden_polygenic_score")
system("gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv burden_polygenic_score")

In [ ]:
# Copy gene-set burden score files to directory 
system("gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/EDI/AoU_EHR_Depth_index_EDI.tsv burden_polygenic_score", intern=TRUE)

In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "ZIP" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v8
dataset_38001642_zip_code_socioeconomic_sql <- paste("
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `zip3_ses_map` zip_code 
    JOIN
        `observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) ) 
        AND observation_source_concept_id = 1585250 
        AND observation.value_as_string NOT LIKE 'Res%'", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
zip_code_socioeconomic_38001642_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "zip_code_socioeconomic_38001642",
  "zip_code_socioeconomic_38001642_*.csv")
message(str_glue('The data will be written to {zip_code_socioeconomic_38001642_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_38001642_zip_code_socioeconomic_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  zip_code_socioeconomic_38001642_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {zip_code_socioeconomic_38001642_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(zip3_as_string = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_38001642_zip_code_socioeconomic_df <- read_bq_export_from_workspace_bucket(zip_code_socioeconomic_38001642_path)

dim(dataset_38001642_zip_code_socioeconomic_df)

head(dataset_38001642_zip_code_socioeconomic_df, 5)

In [ ]:
# ========================================================================================
# Load data for the sensitivity analysis 
# ========================================================================================

# Load phenotype file 
pheno <- fread("burden_polygenic_score/EUR_phenotype_file.txt")
edi <- fread("burden_polygenic_score/AoU_EHR_Depth_index_EDI.tsv")
time <- fread("aml_phenotype_time_to_event_with_covariates.tsv") %>% dplyr::select(-age)
# Format exome burden file 
edi <- rename(edi, 
                "person_id" = "IID")

# Load exome burden file 
exome <- fread("burden_polygenic_score/Exome-Burden-PRS.tsv")
 # Load genomic metric file 
metrics <- fread("burden_polygenic_score/genomic_metrics.tsv")
# Format exome burden file 
exome <- rename(exome, 
                "person_id" = "IID", 
                "exome_burden_prs" = "EXOME_SCORE1_SUM") %>% 
         select(person_id, exome_burden_prs)
# Format genomic metric file 
metrics <- select(rename(metrics, "person_id" = "research_id"), person_id, sample_source)

In [ ]:
# ========================================================================================
# Merge together dataframe into single consolidated dataframe 
# ========================================================================================

# Load gene-set burden score 
score <- fread(paste0("burden_polygenic_score/top_genes_prs.tsv")) 
# Rename column present in score file 
score <- rename(score, "person_id" = "IID") 
# List of data frames
df_list <- list(pheno, score, exome, metrics, edi, time, dataset_38001642_zip_code_socioeconomic_df)

# Merge all data frames 
merged_df <- Reduce(function(x, y) inner_join(x, y, by = "person_id"), df_list)

In [ ]:
# ========================================================================================
# Standardize numerical predictors 
# ========================================================================================

# Scale remaining unscale numeric columns 
merged_df[, c("ENSG00000119772_SCORE1_SUM", 
              "ENSG00000168769_SCORE1_SUM", 
              "ENSG00000161547_SCORE1_SUM", 
              "observation_period", 
              "date_of_birth_year", 
              "observation_period_start_year",
              "age_at_observation_period_start", 
              "exome_burden_prs",
              "assisted_income",
              "high_school_education",
              "median_income",
              "no_health_insurance",
              "poverty",
              "vacant_housing",
              "deprivation_index")] <-
                        as.data.frame(scale(merged_df[, c("ENSG00000119772_SCORE1_SUM", 
                                                          "ENSG00000168769_SCORE1_SUM", 
                                                          "ENSG00000161547_SCORE1_SUM", 
                                                          "observation_period", 
                                                          "date_of_birth_year", 
                                                          "observation_period_start_year",
                                                          "age_at_observation_period_start", 
                                                          "exome_burden_prs",
                                                          "assisted_income",
                                                          "high_school_education",
                                                          "median_income",
                                                          "no_health_insurance",
                                                          "poverty",
                                                          "vacant_housing",
                                                          "deprivation_index")]))

In [ ]:
# List of top genes 
genes <- c("ENSG00000119772", "ENSG00000168769", "ENSG00000161547")

## LOGISTIC REGRESSION LOOP - DEPRIVATION INDEX

In [ ]:
# ========================================================================================
# Logistic Regression Loop - Deprivation Index 
# ========================================================================================
 
coefficients <- list() 

for (gene in genes){
    
    # BASIC LOGISTIC REGRESSION 
    logistic <- 
        glm(paste0("aml_phenotype ~ ",gene,"_SCORE1_SUM + exome_burden_prs + sex + age + 
         PC1 + PC2 + PC3 + PC4 + PC5 + deprivation_index + 
         PC6 + PC7 + PC8 + PC9 + PC10"), 
         data = merged_df, 
         family="binomial")
    # Summary of logistic regression 
    logistic_summary <- summary(logistic)
    # Coefficients of logistic regression 
    logistic_coef <- coef(logistic_summary)
    logistic_coef <- rownames_to_column(as.data.frame(logistic_coef), var = "Covariate")
    
    logistic_coef$model <- rep("Baseline", nrow(logistic_coef))
    logistic_coef$gene <- rep(gene, nrow(logistic_coef))
    
    coefficients[[paste0(gene,"-baseline-model")]] <- logistic_coef
    
    # EDI LOGISTIC REGRESSION 
    logistic <- 
        glm(paste0("aml_phenotype ~ ",gene,"_SCORE1_SUM + exome_burden_prs + sex + age + EDI +
         PC1 + PC2 + PC3 + PC4 + PC5 + deprivation_index + 
         PC6 + PC7 + PC8 + PC9 + PC10"), 
         data = merged_df, 
         family="binomial")
    # Summary of logistic regression 
    logistic_summary <- summary(logistic)
    # Coefficients of logistic regression 
    logistic_coef <- coef(logistic_summary)
    logistic_coef <- rownames_to_column(as.data.frame(logistic_coef), var = "Covariate")
    
    logistic_coef$model <- rep("Baseline with EDI", nrow(logistic_coef))
    logistic_coef$gene <- rep(gene, nrow(logistic_coef))
    
    coefficients[[paste0(gene,"-edi-model")]] <- logistic_coef
    
    # OBSERVATION PERIOD LOGISTIC REGRESSION 
    logistic <- 
        glm(paste0("aml_phenotype ~ ",gene,"_SCORE1_SUM + exome_burden_prs + sex + age + 
         observation_period + deprivation_index + 
         PC1 + PC2 + PC3 + PC4 + PC5 + 
         PC6 + PC7 + PC8 + PC9 + PC10"), 
         data = merged_df, 
         family="binomial")
    # Summary of logistic regression 
    logistic_summary <- summary(logistic)
    # Coefficients of logistic regression 
    logistic_coef <- coef(logistic_summary)
    logistic_coef <- rownames_to_column(as.data.frame(logistic_coef), var = "Covariate")
    
    logistic_coef$model <- rep("Baseline with Observation Period", nrow(logistic_coef))
    logistic_coef$gene <- rep(gene, nrow(logistic_coef))
    
    coefficients[[paste0(gene,"-observation-period-model")]] <- logistic_coef
    
}

In [ ]:
# Combine results 
logistic_coefficients <- do.call(rbind, coefficients)
gene_conversion <- c("ENSG00000119772" = "DNMT3A", 
                     "ENSG00000168769" = "TET2",
                     "ENSG00000161547" = "SRSF2")
logistic_coefficients$gene <- gene_conversion[logistic_coefficients$gene]
logistic_coefficients <- 
    logistic_coefficients %>% 
    clean_names() %>% 
    mutate(covariate = ifelse(grepl("SCORE1_SUM", covariate), gene, covariate))
write_tsv( janitor::clean_names(logistic_coefficients),
          "top_genes_prs_with_deprivation_index_logistic_regression_coefficients.tsv")


In [ ]:
# ========================================================================================
# Logistic Regression Plots - Deprivation Index 
# ========================================================================================

options(repr.plot.height = 15, repr.plot.width = 20)

plot <-
  logistic_coefficients %>% 
  mutate(covariate = gsub("SCORE1_SUM", "", covariate)) %>% 
  mutate(covariate = toupper(covariate)) %>% 
  mutate(covariate = gsub("_", " ", covariate)) %>% 
  mutate(
    upper_ci = estimate + 1.96 * std_error,
    lower_ci = estimate - 1.96 * std_error,
    covariate = factor(covariate, levels = sort(unique(covariate))) 
  ) %>% 
  ggplot(aes(x = estimate, y = covariate)) +
    geom_point() +
    geom_errorbar(aes(xmin = lower_ci, xmax = upper_ci), width = 0.2) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "red") +
    facet_wrap(vars(model, gene), ncol = 3) +
    theme_minimal() +
    theme(axis.text.x = element_text(size = rel(0.7), angle = 25, hjust = 1)) +
    labs(x = "Coefficient", y = "Covariate")

ggsave("top_genes_prs_with_deprivation_index_logistic_regression_coefficients.png",
       plot = plot, 
       height = 15, 
       width = 20, 
       dpi = 300)


options(repr.plot.height = 5, repr.plot.width = 10)

plot <-
 logistic_coefficients %>% 
 mutate(negative_log_p_values = -log10(pr_z)) %>%
 filter(covariate == gene) %>% 
 ggplot(aes(y = negative_log_p_values, x = model)) + 
 geom_segment( 
         aes(x=model, xend=model, y=0, yend=negative_log_p_values), 
         size=1, color="#F7AA14FF", linetype="dotdash") +
 geom_point(size = 3) +
 theme_minimal() + 
 facet_wrap(vars(gene)) + 
 theme(axis.text.x = element_text(size = rel(0.7), angle = 10, hjust = 0.75)) + 
 labs(x = "Model", y = "-Log₁₀(P-values)")

ggsave("top_genes_prs_with_deprivation_index_logistic_regression_pvalues.png",
       plot = plot, 
       height = 5, 
       width = 10, 
       dpi = 300)

## LOGISTIC REGRESSION LOOP - DEPRIVATOIN INDEX COMPONENTS 

In [ ]:
# ========================================================================================
# Logistic Regression Loop - Deprivation Index Components 
# ========================================================================================
 
coefficients <- list() 

for (gene in genes){
    
    # BASIC LOGISTIC REGRESSION 
    logistic <- 
        glm(paste0("aml_phenotype ~ ",gene,"_SCORE1_SUM + exome_burden_prs + sex + age + 
         PC1 + PC2 + PC3 + PC4 + PC5 + assisted_income + high_school_education + median_income + no_health_insurance + poverty + vacant_housing + 
         PC6 + PC7 + PC8 + PC9 + PC10"), 
         data = merged_df, 
         family="binomial")
    # Summary of logistic regression 
    logistic_summary <- summary(logistic)
    # Coefficients of logistic regression 
    logistic_coef <- coef(logistic_summary)
    logistic_coef <- rownames_to_column(as.data.frame(logistic_coef), var = "Covariate")
    
    logistic_coef$model <- rep("Baseline", nrow(logistic_coef))
    logistic_coef$gene <- rep(gene, nrow(logistic_coef))
    
    coefficients[[paste0(gene,"-baseline-model")]] <- logistic_coef
    
    # EDI LOGISTIC REGRESSION 
    logistic <- 
        glm(paste0("aml_phenotype ~ ",gene,"_SCORE1_SUM + exome_burden_prs + sex + age + EDI +
         PC1 + PC2 + PC3 + PC4 + PC5 + assisted_income + high_school_education + median_income + no_health_insurance + poverty + vacant_housing + 
         PC6 + PC7 + PC8 + PC9 + PC10"), 
         data = merged_df, 
         family="binomial")
    # Summary of logistic regression 
    logistic_summary <- summary(logistic)
    # Coefficients of logistic regression 
    logistic_coef <- coef(logistic_summary)
    logistic_coef <- rownames_to_column(as.data.frame(logistic_coef), var = "Covariate")
    
    logistic_coef$model <- rep("Baseline with EDI", nrow(logistic_coef))
    logistic_coef$gene <- rep(gene, nrow(logistic_coef))
    
    coefficients[[paste0(gene,"-edi-model")]] <- logistic_coef
    
    # OBSERVATION PERIOD LOGISTIC REGRESSION 
    logistic <- 
        glm(paste0("aml_phenotype ~ ",gene,"_SCORE1_SUM + exome_burden_prs + sex + age + 
         observation_period + assisted_income + high_school_education + median_income + no_health_insurance + poverty + vacant_housing + 
         PC1 + PC2 + PC3 + PC4 + PC5 + 
         PC6 + PC7 + PC8 + PC9 + PC10"), 
         data = merged_df, 
         family="binomial")
    # Summary of logistic regression 
    logistic_summary <- summary(logistic)
    # Coefficients of logistic regression 
    logistic_coef <- coef(logistic_summary)
    logistic_coef <- rownames_to_column(as.data.frame(logistic_coef), var = "Covariate")
    
    logistic_coef$model <- rep("Baseline with Observation Period", nrow(logistic_coef))
    logistic_coef$gene <- rep(gene, nrow(logistic_coef))
    
    coefficients[[paste0(gene,"-observation-period-model")]] <- logistic_coef
    
}

In [ ]:
# Combine results 
logistic_coefficients <- do.call(rbind, coefficients)
gene_conversion <- c("ENSG00000119772" = "DNMT3A", 
                     "ENSG00000168769" = "TET2",
                     "ENSG00000161547" = "SRSF2")
logistic_coefficients$gene <- gene_conversion[logistic_coefficients$gene]
logistic_coefficients <- 
    logistic_coefficients %>% 
    clean_names() %>% 
    mutate(covariate = ifelse(grepl("SCORE1_SUM", covariate), gene, covariate))
write_tsv( janitor::clean_names(logistic_coefficients),
          "top_genes_prs_with_deprivation_components_logistic_regression_coefficients.tsv")

In [ ]:
# ========================================================================================
# Logistic Regression Plots - Deprivation Index Components 
# ========================================================================================

options(repr.plot.height = 15, repr.plot.width = 20)

plot <-
  logistic_coefficients %>% 
  mutate(covariate = gsub("SCORE1_SUM", "", covariate)) %>% 
  mutate(covariate = toupper(covariate)) %>% 
  mutate(covariate = gsub("_", " ", covariate)) %>% 
  mutate(
    upper_ci = estimate + 1.96 * std_error,
    lower_ci = estimate - 1.96 * std_error,
    covariate = factor(covariate, levels = sort(unique(covariate))) 
  ) %>% 
  ggplot(aes(x = estimate, y = covariate)) +
    geom_point() +
    geom_errorbar(aes(xmin = lower_ci, xmax = upper_ci), width = 0.2) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "red") +
    facet_wrap(vars(model, gene), ncol = 3) +
    theme_minimal() +
    theme(axis.text.x = element_text(size = rel(0.7), angle = 25, hjust = 1)) +
    labs(x = "Coefficient", y = "Covariate")

ggsave("top_genes_prs_with_deprivation_components_logistic_regression_coefficients.png",
       plot = plot, 
       height = 15, 
       width = 20, 
       dpi = 300)


options(repr.plot.height = 5, repr.plot.width = 10)

plot <-
   logistic_coefficients %>% 
   mutate(negative_log_p_values = -log10(pr_z)) %>%
   filter(covariate == gene) %>% 
   ggplot(aes(y = negative_log_p_values, x = model)) + 
   geom_segment( 
         aes(x=model, xend=model, y=0, yend=negative_log_p_values), 
         size=1, color="#F7AA14FF", linetype="dotdash") +
   geom_point(size = 3) +
   theme_minimal() + 
   facet_wrap(vars(gene)) + 
   theme(axis.text.x = element_text(size = rel(0.7), angle = 10, hjust = 0.75)) + 
   labs(x = "Model", y = "-Log₁₀(P-values)")

ggsave("top_genes_prs_with_deprivation_components_logistic_regression_pvalues.png",
       plot = plot, 
       height = 5, 
       width = 10, 
       dpi = 300)

## COX REGRESSION LOOP - DEPRIVATION INDEX 

In [ ]:
# ========================================================================================
# Cox Regression Loop - Deprivation Index 
# ========================================================================================

coefficients <- list() 

for (gene in genes){

    # COX REGRESSION 
    cox_formula <- as.formula(paste0("Surv(time_to_event_days_from_birth, status) ~ ",
                                     gene, "_SCORE1_SUM + exome_burden_prs + sex + date_of_birth_year +",
                                     "high_school_education + median_income + no_health_insurance + poverty + vacant_housing +",
                                     "PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")
                             )
    # Run cox regression 
    cox_model <- coxph(cox_formula, data = merged_df)
    cox_summary <- summary(cox_model)
    # Coefficients of cox regression 
    cox_coef <- coef(cox_summary)
    cox_coef <- rownames_to_column(as.data.frame(cox_coef), var = "Covariate")
    
    cox_coef$model <- rep("Time from Brith", nrow(cox_coef))
    cox_coef$gene <- rep(gene, nrow(cox_coef))
    
    coefficients[[paste0(gene,"-time-from-birth-model")]] <- cox_coef
    
    # COX REGRESSION 
    cox_formula <- as.formula(paste0("Surv(time_to_event_days_from_observation, status) ~ ",
                                     gene, "_SCORE1_SUM + exome_burden_prs + sex +",
                                     "observation_period_start_year + age_at_observation_period_start +",
                                     "high_school_education + median_income + no_health_insurance + poverty + vacant_housing +",
                                     "PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")
                             )
    # Run cox regression 
    cox_model <- coxph(cox_formula, data = merged_df)
    cox_summary <- summary(cox_model)
    # Coefficients of cox regression 
    cox_coef <- coef(cox_summary)
    cox_coef <- rownames_to_column(as.data.frame(cox_coef), var = "Covariate")
    
    cox_coef$model <- rep("Time from Observation Period", nrow(cox_coef))
    cox_coef$gene <- rep(gene, nrow(cox_coef))
    
    coefficients[[paste0(gene,"-time-from-observation-period-model")]] <- cox_coef
    
    # EDI COX REGRESSION 
    cox_formula <- as.formula(paste0("Surv(time_to_event_days_from_observation, status) ~ ",
                                     gene, "_SCORE1_SUM + exome_burden_prs + sex + EDI +",
                                     "observation_period_start_year + age_at_observation_period_start +",
                                     "high_school_education + median_income + no_health_insurance + poverty + vacant_housing +",
                                     "PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")
                             )
    # Run cox regression 
    cox_model <- coxph(cox_formula, data = merged_df)
    cox_summary <- summary(cox_model)
    # Coefficients of cox regression 
    cox_coef <- coef(cox_summary)
    cox_coef <- rownames_to_column(as.data.frame(cox_coef), var = "Covariate")
    
    cox_coef$model <- rep("Time from Observation Period with EDI", nrow(cox_coef))
    cox_coef$gene <- rep(gene, nrow(cox_coef))
    
    coefficients[[paste0(gene,"-time-from-observation-period_with-edi-model")]] <- cox_coef
    
}

In [ ]:
# Combine results 
cox_coefficients <- do.call(rbind, coefficients)
gene_conversion <- c("ENSG00000119772" = "DNMT3A", 
                     "ENSG00000168769" = "TET2",
                     "ENSG00000161547" = "SRSF2")
cox_coefficients$gene <- gene_conversion[cox_coefficients$gene]
cox_coefficients <- 
    cox_coefficients %>% 
    clean_names() %>% 
    mutate(covariate = ifelse(grepl("SCORE1_SUM", covariate), gene, covariate))
write_tsv(janitor::clean_names(cox_coefficients),
          "top_genes_prs_with_deprivation_index_cox_regression_coefficients.tsv")

In [ ]:
# ========================================================================================
# Cox Regression Plots - Deprivation Index 
# ========================================================================================

options(repr.plot.height = 15, repr.plot.width = 20)

plot <- 
  cox_coefficients %>% 
  mutate(covariate = gsub("SCORE1_SUM", "", covariate)) %>% 
  mutate(covariate = toupper(covariate)) %>% 
  mutate(covariate = gsub("_", " ", covariate)) %>% 
  mutate(
    upper_ci = coef + 1.96 * se_coef,
    lower_ci = coef - 1.96 * se_coef,
    covariate = factor(covariate, levels = sort(unique(covariate))) 
  ) %>% 
  ggplot(aes(x = coef, y = covariate)) +
    geom_point() +
    geom_errorbar(aes(xmin = lower_ci, xmax = upper_ci), width = 0.2) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "red") +
    facet_wrap(vars(model, gene), ncol = 3) +
    theme_minimal() +
    theme(axis.text.x = element_text(size = rel(0.7), angle = 25, hjust = 1)) +
    labs(x = "Coefficient", y = "Covariate")

ggsave("top_genes_prs_with_deprivation_index_cox_regression_coefficients.png",
       plot = plot, 
       height = 15, 
       width = 20, 
       dpi = 300)

options(repr.plot.height = 5, repr.plot.width = 10)

plot <- 
    cox_coefficients %>% 
    mutate(negative_log_p_values = -log10(pr_z)) %>%
    filter(covariate == gene) %>% 
    ggplot(aes(y = negative_log_p_values, x = model)) + 
    geom_segment( 
         aes(x=model, xend=model, y=0, yend=negative_log_p_values), 
         size=1, color="#F7AA14FF", linetype="dotdash") +
    geom_point(size = 3) +
    theme_minimal() + 
    facet_wrap(vars(gene)) + 
    theme(axis.text.x = element_text(size = rel(0.7), angle = 10, hjust = 0.75)) + 
    labs(x = "Model", y = "-Log₁₀(P-values)")

ggsave("top_genes_prs_with_deprivation_index_cox_regression_pvalues.png",
       plot = plot, 
       height = 5, 
       width = 10, 
       dpi = 300)


## COX REGRESSSION LOOP - DEPRIVATION INDEX COMPONENTS 

In [ ]:
# ========================================================================================
# Cox Regression Loop - Deprivation Index Components 
# ========================================================================================

coefficients <- list() 

for (gene in genes){

    # COX REGRESSION 
    cox_formula <- as.formula(paste0("Surv(time_to_event_days_from_birth, status) ~ ",
                                     gene, "_SCORE1_SUM + exome_burden_prs + sex + date_of_birth_year +",
                                     "deprivation_index +  ",
                                     "PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")
                             )
    # Run cox regression 
    cox_model <- coxph(cox_formula, data = merged_df)
    cox_summary <- summary(cox_model)
    # Coefficients of cox regression 
    cox_coef <- coef(cox_summary)
    cox_coef <- rownames_to_column(as.data.frame(cox_coef), var = "Covariate")
    
    cox_coef$model <- rep("Time from Brith", nrow(cox_coef))
    cox_coef$gene <- rep(gene, nrow(cox_coef))
    
    coefficients[[paste0(gene,"-time-from-birth-model")]] <- cox_coef
    
    # COX REGRESSION 
    cox_formula <- as.formula(paste0("Surv(time_to_event_days_from_observation, status) ~ ",
                                     gene, "_SCORE1_SUM + exome_burden_prs + sex +",
                                     "observation_period_start_year + age_at_observation_period_start +",
                                     "deprivation_index + ",
                                     "PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")
                             )
    # Run cox regression 
    cox_model <- coxph(cox_formula, data = merged_df)
    cox_summary <- summary(cox_model)
    # Coefficients of cox regression 
    cox_coef <- coef(cox_summary)
    cox_coef <- rownames_to_column(as.data.frame(cox_coef), var = "Covariate")
    
    cox_coef$model <- rep("Time from Observation Period", nrow(cox_coef))
    cox_coef$gene <- rep(gene, nrow(cox_coef))
    
    coefficients[[paste0(gene,"-time-from-observation-period-model")]] <- cox_coef
    
    # EDI COX REGRESSION 
    cox_formula <- as.formula(paste0("Surv(time_to_event_days_from_observation, status) ~ ",
                                     gene, "_SCORE1_SUM + exome_burden_prs + sex + EDI +",
                                     "observation_period_start_year + age_at_observation_period_start +",
                                     "deprivation_index + ",
                                     "PC1 + PC2 + PC3 + PC4 + PC5 + PC6 + PC7 + PC8 + PC9 + PC10")
                             )
    # Run cox regression 
    cox_model <- coxph(cox_formula, data = merged_df)
    cox_summary <- summary(cox_model)
    # Coefficients of cox regression 
    cox_coef <- coef(cox_summary)
    cox_coef <- rownames_to_column(as.data.frame(cox_coef), var = "Covariate")
    
    cox_coef$model <- rep("Time from Observation Period with EDI", nrow(cox_coef))
    cox_coef$gene <- rep(gene, nrow(cox_coef))
    
    coefficients[[paste0(gene,"-time-from-observation-period_with-edi-model")]] <- cox_coef
 
}

In [ ]:
# Combine results 
cox_coefficients <- do.call(rbind, coefficients)
gene_conversion <- c("ENSG00000119772" = "DNMT3A", 
                     "ENSG00000168769" = "TET2",
                     "ENSG00000161547" = "SRSF2")
cox_coefficients$gene <- gene_conversion[cox_coefficients$gene]
cox_coefficients <- 
    cox_coefficients %>% 
    clean_names() %>% 
    mutate(covariate = ifelse(grepl("SCORE1_SUM", covariate), gene, covariate))
write_tsv(janitor::clean_names(cox_coefficients),
          "top_genes_prs_with_deprivation_components_cox_regression_coefficients.tsv")

In [ ]:
# ========================================================================================
# Cox Regression Plots - Deprivation Index Components 
# ========================================================================================

options(repr.plot.height = 15, repr.plot.width = 20)

plot <- 
  cox_coefficients %>% 
  mutate(covariate = gsub("SCORE1_SUM", "", covariate)) %>% 
  mutate(covariate = toupper(covariate)) %>% 
  mutate(covariate = gsub("_", " ", covariate)) %>% 
  mutate(
    upper_ci = coef + 1.96 * se_coef,
    lower_ci = coef - 1.96 * se_coef,
    covariate = factor(covariate, levels = sort(unique(covariate))) 
  ) %>% 
  ggplot(aes(x = coef, y = covariate)) +
    geom_point() +
    geom_errorbar(aes(xmin = lower_ci, xmax = upper_ci), width = 0.2) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "red") +
    facet_wrap(vars(model, gene), ncol = 3) +
    theme_minimal() +
    theme(axis.text.x = element_text(size = rel(0.7), angle = 25, hjust = 1)) +
    labs(x = "Coefficient", y = "Covariate")

ggsave("top_genes_prs_with_deprivation_components_cox_regression_coefficients.png",
       plot = plot, 
       height = 15, 
       width = 20, 
       dpi = 300)

options(repr.plot.height = 5, repr.plot.width = 10)

plot <- 
    cox_coefficients %>% 
    mutate(negative_log_p_values = -log10(pr_z)) %>%
    filter(covariate == gene) %>% 
    ggplot(aes(y = negative_log_p_values, x = model)) + 
    geom_segment( 
         aes(x=model, xend=model, y=0, yend=negative_log_p_values), 
         size=1, color="#F7AA14FF", linetype="dotdash") +
    geom_point(size = 3) +
    theme_minimal() + 
    facet_wrap(vars(gene)) + 
    theme(axis.text.x = element_text(size = rel(0.7), angle = 10, hjust = 0.75)) + 
    labs(x = "Model", y = "-Log₁₀(P-values)")

ggsave("top_genes_prs_with_deprivation_components_cox_regression_pvalues.png",
       plot = plot, 
       height = 5, 
       width = 10, 
       dpi = 300)